In [4]:
from ultralytics import YOLO
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import cv2
import random
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

In [6]:
BASE_PATH = "../BoneFractureYolo8"

train_images = os.path.join(BASE_PATH, "train", "images")
train_labels = os.path.join(BASE_PATH, "train", "labels")

val_images = os.path.join(BASE_PATH, "valid", "images")
val_labels = os.path.join(BASE_PATH, "valid", "labels")

test_images = os.path.join(BASE_PATH, "test", "images")
test_labels = os.path.join(BASE_PATH, "test", "labels")

In [8]:
import yaml

with open(os.path.join(BASE_PATH, "data.yaml")) as f:
    data = yaml.safe_load(f)

print(data)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus', 'shoulder fracture', 'wrist positive'], 'roboflow': {'workspace': 'veda', 'project': 'bone-fracture-detection-daoon', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/veda/bone-fracture-detection-daoon/dataset/4'}}


In [12]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=os.path.join(BASE_PATH, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.21  Python-3.12.7 torch-2.5.1+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
run_dir = model.trainer.save_dir
print("Training results saved in:", run_dir)

In [ ]:
metrics = model.val(
    data=os.path.join(BASE_PATH, "data.yaml"),
    split="test"
)

print(metrics)

In [ ]:
results = model.predict(
    source=test_images,
    conf=0.25,
    save=True
)

In [ ]:
conf_matrix_path = os.path.join(run_dir, "confusion_matrix.png")

if os.path.exists(conf_matrix_path):
    img = mpimg.imread(conf_matrix_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()
else:
    print("confusion_matrix.png not found.")

In [ ]:
conf_matrix_norm_path = os.path.join(run_dir, "confusion_matrix_normalized.png")

if os.path.exists(conf_matrix_norm_path):
    img = mpimg.imread(conf_matrix_norm_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Normalized Confusion Matrix")
    plt.show()
else:
    print("confusion_matrix_normalized.png not found.")

In [ ]:
import pandas as pd

results_csv_path = os.path.join(run_dir, "results.csv")

if os.path.exists(results_csv_path):
    df_results = pd.read_csv(results_csv_path)
    print(df_results.columns.tolist())
    display(df_results.head())
else:
    print("results.csv not found.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: losses
if "train/box_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["train/box_loss"], label="train box")
if "train/cls_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["train/cls_loss"], label="train cls")
if "train/dfl_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["train/dfl_loss"], label="train dfl")
if "val/box_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["val/box_loss"], label="val box", linestyle="--")
if "val/cls_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["val/cls_loss"], label="val cls", linestyle="--")
if "val/dfl_loss" in df_results.columns:
    axes[0].plot(df_results["epoch"], df_results["val/dfl_loss"], label="val dfl", linestyle="--")

axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Right: metrics
if "metrics/precision(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/precision(B)"], label="precision")
if "metrics/recall(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/recall(B)"], label="recall")
if "metrics/mAP50(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/mAP50(B)"], label="mAP50")
if "metrics/mAP50-95(B)" in df_results.columns:
    axes[1].plot(df_results["epoch"], df_results["metrics/mAP50-95(B)"], label="mAP50-95")

axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Value")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()